In [12]:
library(terra)
library(data.table)
library(dplyr)
library(stringr)
library(jsonlite)
library(tidyr)
library(data.table)


Attaching package: ‘tidyr’


The following object is masked from ‘package:terra’:

    extract




In [ ]:
# --- Paths
# old trait
trait_dir <- "other/Flux/EcoRes/EcoRes/glob_trait/data/1km/analysis-ready"
# hydraulic
trait_dir <- "panops/panops-data-registry/data/flux/plant_trait/hydraulic"
# EFP location
meta_file <- "/mnt/gsdata/projects/panops/panops-data-registry/data/flux/348EFP_Locations.csv"
###
meta_file <- "panops/panops-data-registry/data/flux/derived_tables/outputs_afterEGU_results/EFP_mortality_combined.csv"
### Migliavaca
meta_file <- "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/MigliavaccaEcosystemfunctionsReprWorkflow/data/InputData_withPCs_Migliavacca2021.csv"

In [7]:
meta_file <- "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/fluxnet_forest_survey.csv"
trait_dir <- "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/glob_trait/data/1km/analysis-ready"

In [8]:
#--- Load tower metadata
towermeta <- fread(meta_file)

# --- List all trait rasters
trait_files <- list.files(trait_dir, pattern = "\\.tif$", full.names = TRUE)

# Helper: extract trait code from filename (first part before "_")
get_trait_code <- function(f) {
  basename(f) |> str_extract("^[^_]+")  # e.g. "X237"
}
trait_codes <- sapply(trait_files, get_trait_code)

# --- Create SpatVector of tower locations (EPSG:4326)
#tower_pts <- vect(towermeta, geom = c("LOCATION_LONG", "LOCATION_LAT"), crs = "EPSG:4326")
tower_pts <- vect(towermeta, geom = c("lon", "lat"), crs = "EPSG:4326")

# --- Reproject tower points to raster CRS (EPSG:6933)
r <- rast(trait_files[1])
tower_pts_proj <- project(tower_pts, crs(r))

# --- Extract values (band 1 = mean, 2 = cv, 3 = applicability)
trait_values <- lapply(seq_along(trait_files), function(i) {
  r <- rast(trait_files[i])
  vals <- terra::extract(r, tower_pts_proj)[, -1]  # drop ID column
  colnames(vals) <- paste0(trait_codes[i], c("_mean", "_cv", "_applic"))
  return(vals)
})

In [13]:
# --- Combine all trait values into one data frame
trait_df <- do.call(cbind, trait_values)

final_df <- cbind(
  towermeta,
  trait_df
)
final_df_mean <- final_df %>%
  dplyr::select(-ends_with("_cv"), -ends_with("_applic"))

 df_long <- final_df_mean %>%
  pivot_longer(
    cols = starts_with("X"),   # all trait columns (assuming they start with X…)
    names_to = "Trait",
    values_to = "Value"
  )

# --- Load your lookup JSON
trait_info <- fromJSON("/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/clean_data/trait_lookup.json")

# Convert into a data.table for easier mapping
trait_map <- data.table(
  code = names(trait_info),
  short = sapply(trait_info, function(x) x$short),
  long = sapply(trait_info, function(x) x$long),
  unit = sapply(trait_info, function(x) x$unit)
)


# Add "X" prefix to code column in trait_map
trait_map <- trait_map %>%
  mutate(codeX = paste0("X", code))

trait_map_mean <- trait_map %>%
  mutate(codeX_mean = paste0(codeX, "_mean"))
# Join df_long with trait_map to get short/long names
df_long_named <- df_long %>%
  left_join(trait_map_mean, by = c("Trait" = "codeX_mean"))

## rename the short column with trait_name
df_long_named <- df_long_named %>%
  rename(trait_name = short)


df_wide_named <- df_long_named %>%
  tibble::as_tibble() %>%
  dplyr::distinct() %>%
  tidyr::pivot_wider(
    id_cols = c(1:13),#SITE_ID:forest_mean_500m,   # all site metadata
    names_from = trait_name,
    values_from = Value,
    values_fn = list(Value = mean)     # collapse duplicates
  )


In [14]:
View(df_wide_named)

site_id,site_name,igbp,lat,lon,record_start,record_end,record_years,has_archive,in_basfor_metadata,⋯,Leaf area (3112),Leaf area,Leaf area (3114),SLA,SSD,Leaf thickness,Leaf N (area),Leaf dry mass,Rooting depth,Leaf delta 15N
<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<lgl>,<lgl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AR-SLu,San Luis,MF,-33.4648,-66.4598,2009,2011,3,TRUE,FALSE,⋯,504.0892,602.6079,420.4412,9.927664,0.5642032,0.2711317,1.922857,53.15581,0.6918347,1.80658666
AR-Vir,Virasoro,ENF,-28.2395,-56.1886,2009,2012,4,TRUE,FALSE,⋯,1744.3706,1408.6049,1088.5145,14.662185,0.4722769,0.2042549,1.432450,107.23314,1.0653660,0.71522217
AU-Cum,Cumberland Plain,EBF,-33.6152,150.7236,2012,2014,3,TRUE,TRUE,⋯,627.4383,532.2699,496.4719,11.565023,0.5413988,0.2450193,1.716706,58.95957,0.7937069,0.12485496
AU-Lox,Loxton,DBF,-34.4704,140.6551,2008,2009,2,TRUE,FALSE,⋯,266.4440,241.2823,236.7210,10.820020,0.5089876,0.2751936,2.341995,23.38597,0.6691964,1.52822563
AU-Rob,"Robson Creek, Queensland, Australia",EBF,-17.1175,145.6301,2014,2014,1,TRUE,TRUE,⋯,2907.7002,2534.0137,2931.2004,12.093115,0.5568957,0.2358074,1.492418,228.90713,1.5520886,0.93470419
AU-Tum,Tumbarumba,EBF,-35.6566,148.1517,2001,2014,14,TRUE,TRUE,⋯,321.8946,401.7109,301.0355,13.798202,0.4741352,0.2331236,1.514656,26.73166,0.3635799,-0.44527259
AU-Wac,Wallaby Creek,EBF,-37.4259,145.1878,2005,2008,4,TRUE,FALSE,⋯,545.9600,410.8644,446.8650,12.334738,0.4760566,0.2543037,1.515476,44.82572,0.5673243,-0.67080023
AU-Whr,Whroo,EBF,-36.6732,145.0294,2011,2014,4,TRUE,TRUE,⋯,350.1857,312.1021,289.0701,8.531928,0.5756054,0.2735979,1.929146,33.83273,0.7597495,-0.02943959
AU-Wom,Wombat,EBF,-37.4222,144.0944,2010,2014,5,TRUE,TRUE,⋯,419.2159,390.1484,304.0269,11.384722,0.4897266,0.2498791,1.744958,38.47573,0.5786434,-0.40505613


In [ ]:
#### write the file
fwrite(df_wide_named, "panops/panops-data-registry/data/flux/derived_tables/outputs_afterEGU_results/EFP_mortality_trait_combined.csv")

In [16]:
old_traits <- df_wide_named

Hydraulic trait

In [19]:
meta_file <- "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/fluxnet_forest_survey.csv"
trait_dir <- "/mnt/gsdata/projects/panops/panops-data-registry/data/flux/plant_trait/hydraulic"

In [20]:
#--- Load tower metadata
towermeta <- fread(meta_file)
# --- List all trait rasters
trait_files <- list.files(trait_dir, pattern = "\\.tif$", full.names = TRUE)

# Helper: extract trait code from filename (first part before "_")
get_trait_code <- function(f) {
  basename(f) |> str_extract("^[^_]+")  # e.g. "X237"
}
trait_codes <- sapply(trait_files, get_trait_code)


# --- Create SpatVector of tower locations (EPSG:4326)
#tower_pts <- vect(towermeta, geom = c("LOCATION_LONG", "LOCATION_LAT"), crs = "EPSG:4326")
tower_pts <- vect(towermeta, geom = c("lon", "lat"), crs = "EPSG:4326")
# --- Use first raster as CRS reference and reproject points
r0 <- rast(trait_files[1])
tower_pts_proj <- project(tower_pts, crs(r0))

# --- Extract values (single band per raster)
trait_values <- lapply(seq_along(trait_files), function(i) {
  r <- rast(trait_files[i])
  
  # safety: if a raster has more than 1 layer, keep only the first
  if (nlyr(r) > 1) r <- r[[1]]
  
  vals <- terra::extract(r, tower_pts_proj)[, 2, drop = FALSE]  # keep as 1-col table
  colnames(vals) <- paste0(trait_codes[i], "_mean")            # or just trait_codes[i]
  vals
})

# --- Combine all trait values into one data frame
trait_df <- do.call(cbind, trait_values)    

final_df <- cbind(
  df_wide_named,
  trait_df
)



In [22]:
#### write the file
fwrite(final_df, "/mnt/gsdata/projects/other/Flux/EcoRes/EcoRes/fluxnet_forest_survey_trait_combined.csv")

In [21]:
str(final_df)


'data.frame':	101 obs. of  36 variables:
 $ site_id                    : chr  "AR-SLu" "AR-Vir" "AU-Cum" "AU-Lox" ...
 $ site_name                  : chr  "San Luis" "Virasoro" "Cumberland Plain" "Loxton" ...
 $ igbp                       : chr  "MF" "ENF" "EBF" "DBF" ...
 $ lat                        : num  -33.5 -28.2 -33.6 -34.5 -17.1 ...
 $ lon                        : num  -66.5 -56.2 150.7 140.7 145.6 ...
 $ record_start               : int  2009 2009 2012 2008 2014 2001 2005 2011 2010 1996 ...
 $ record_end                 : int  2011 2012 2014 2009 2014 2014 2008 2014 2014 2014 ...
 $ record_years               : int  3 4 3 2 1 14 4 4 5 19 ...
 $ has_archive                : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
 $ in_basfor_metadata         : logi  FALSE FALSE TRUE FALSE TRUE TRUE ...
 $ climate_koeppen            : chr  "" "" "Cfa" "" ...
 $ climate_koeppen_description: chr  "" "" "Humid subtropical, no dry season, hot summer" "" ...
 $ notes                      : chr  "" 